In [15]:
import sys
import os
import re
sys.path.append(os.path.abspath('..'))

from prompts.clustering import build_clustering_prompt_for_algorithm
from prompts.association_rule import build_association_rule_prompt
from prompts.time_series import build_time_series_prompt

from utils.read_file import read_file
from models.agent.groq import GroqModel

In [39]:
files = [
    "data/Iris.csv",
    "data/Mall_Customers.csv",
    "data/online_retail_II.csv",
    "data/synthetic_transactions.csv",
    "data/Electric_Production.csv",
]

In [11]:
preprocessed_code = []
model = GroqModel()

GroqModel initialized with Groq client.


In [16]:
dataframe = read_file(files[0])

prompt = build_clustering_prompt_for_algorithm(dataframe, "k-means, dbscan, hierarchical")

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


In [19]:
dataframe = read_file(files[1])

prompt = build_clustering_prompt_for_algorithm(dataframe, "k-means, dbscan, hierarchical")

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


In [20]:
dataframe = read_file(files[2])

prompt = build_association_rule_prompt(dataframe)

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


In [21]:
dataframe = read_file(files[3])

prompt = build_association_rule_prompt(dataframe)

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


In [ ]:
dataframe = read_file(files[3])

prompt = build_association_rule_prompt(dataframe)

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


In [46]:
dataframe = read_file(files[4])

prompt = build_association_rule_prompt(dataframe)

code = model.generate(prompt=prompt)

code = re.sub(r"^```(?:python)?\s*", "", code.strip(), flags=re.MULTILINE)
code = re.sub(r"```$", "", code.strip(), flags=re.MULTILINE)

preprocessed_code.append(code)


| Check                             | Pass/Fail |
| --------------------------------- | --------- |
| Handles missing values            |   Not Have|
| Encodes categorical columns       |   Not Have|
| Scales numerical features         |   Pass    |
| Removes ID columns                |   Pass    |
| Avoids target leakage             |   Pass    |
| Handles datetime columns properly |   Not Have|


In [29]:
print(preprocessed_code[0])

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(input_path)
df = df.drop(columns=['Id', 'Species'])
scaler = StandardScaler()
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)


| Check                             | Pass/Fail |
| --------------------------------- | --------- |
| Handles missing values            |   Pass    |
| Encodes categorical columns       |   Pass    |
| Scales numerical features         |   Pass    |
| Removes ID columns                |   Pass    |
| Avoids target leakage             |   Not Have|
| Handles datetime columns properly |   Not Have|


In [30]:
print(preprocessed_code[1])

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(input_path)
df = df.drop(columns=['CustomerID'])
df['Genre'] = df['Genre'].map({'Male': 0, 'Female': 1})
df = df.fillna(df.mean())
scaler = StandardScaler()
df[df.columns] = scaler.fit_transform(df)


| Check                             | Pass/Fail |
| --------------------------------- | --------- |
| Handles missing values            |   Pass    |
| Encodes categorical columns       |   Not Have|
| Scales numerical features         |   Not Have|
| Removes ID columns                |   Not Have|
| Avoids target leakage             |   Not Have|
| Handles datetime columns properly |   Not Have|
| Handle items for association rule |   Pass    |


In [27]:
print(preprocessed_code[2])

import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv(input_path)
df = df.dropna(subset=['StockCode'])
basket = df.groupby('Invoice')['StockCode'].apply(lambda x: list(set(x))).reset_index()
mlb = MultiLabelBinarizer()
one_hot = pd.DataFrame(mlb.fit_transform(basket['StockCode']), columns=mlb.classes_, index=basket['Invoice'])
df = one_hot.astype(int)


| Check                             | Pass/Fail |
| --------------------------------- | --------- |
| Handles missing values            |   Pass    |
| Encodes categorical columns       |   Not Have|
| Scales numerical features         |   Not Have|
| Removes ID columns                |   Not Have|
| Avoids target leakage             |   Not Have|
| Handles datetime columns properly |   Not Have|
| Handle items for association rule |   Pass    |


In [28]:
print(preprocessed_code[3])

df = pd.read_csv(input_path)
items_series = df['Items'].fillna('').apply(lambda x: [i.strip() for i in x.split(',') if i.strip()!=''])
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
binary_matrix = mlb.fit_transform(items_series)
df = pd.DataFrame(binary_matrix, columns=mlb.classes_, index=df.index)


| Check                             | Pass/Fail |
| --------------------------------- | --------- |
| Handles missing values            |   Not Have|
| Encodes categorical columns       |   Not Have|
| Scales numerical features         |   Not Have|
| Removes ID columns                |   Not Have|
| Avoids target leakage             |   Not Have|
| Handles datetime columns properly |   Pass    |


In [47]:
print(preprocessed_code[4])

import pandas as pd
df = pd.read_csv(input_path)
bins = pd.cut(df['IPG2211A2N'], bins=5, labels=[f'Bin_{i}' for i in range(5)], include_lowest=True)
df = pd.get_dummies(bins)
